In [28]:
!pip -q install google-genai

In [29]:
import json
import importlib.util
from google.colab import drive
from google import genai

In [30]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
PROJECT_PATH = "/content/drive/MyDrive/VoiceVision-AI"

KNOWLEDGE_BASE_PATH = f"{PROJECT_PATH}/04_Project_Code/knowledge_base.py"

In [32]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [33]:
client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini Connected Successfully!")

Gemini Connected Successfully!


In [34]:
spec = importlib.util.spec_from_file_location(
    "knowledge_base",
    KNOWLEDGE_BASE_PATH
)

knowledge_module = importlib.util.module_from_spec(spec)

spec.loader.exec_module(knowledge_module)

knowledge = knowledge_module.knowledge

print("Knowledge Base Loaded Successfully!")
print("Total Classes:", len(knowledge))

Knowledge Base Loaded Successfully!
Total Classes: 12


In [35]:
def build_prompt(item, question):

    info = knowledge[item]

    prompt = f"""
You are an intelligent waste management assistant.

Detected Waste Item:
{item}

Knowledge:

Material:
{info["Material"]}

Recyclable:
{info["Recyclable"]}

Recommended Bin:
{info["Bin"]}

Environmental Impact:
{info["Environment"]}

Reuse:
{info["Reuse"]}

Decomposition:
{info["Decomposition"]}

Tips:
{", ".join(info["Tips"])}

Warnings:
{", ".join(info["Warnings"])}

User Question:

{question}

Instructions:

1. Answer naturally.
2. Use ONLY the knowledge above.
3. Do not invent facts.
4. Answer in less than 120 words.
5. Be friendly.
"""

    return prompt

In [36]:
def ask_gemini(prompt):

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [37]:
item = "Plastic"

question = "Can I recycle this bottle?"

prompt = build_prompt(item, question)

print(prompt)


You are an intelligent waste management assistant.

Detected Waste Item:
Plastic

Knowledge:

Material:
Plastic is a synthetic polymer derived primarily from petroleum or natural gas through chemical processing. It is generally non-biodegradable, though recyclability varies depending on the specific polymer type. Recycled plastic can be reprocessed into new products, though quality often degrades with repeated recycling cycles.

Recyclable:
True

Recommended Bin:
Blue Recycling Bin

Environmental Impact:
Improperly discarded plastic waste poses severe threats to marine and terrestrial wildlife through ingestion and entanglement. Plastics break down into microplastics that contaminate soil, waterways, and food chains, with long-lasting ecological consequences. Reducing plastic waste and improving recycling rates are critical to mitigating widespread plastic pollution.

Reuse:
Plastic containers can often be reused for storage, organization, or as planters after cleaning. Certain plasti

In [38]:
answer = ask_gemini(prompt)

print(answer)

Yes, you can likely recycle this plastic bottle! Plastic is generally recyclable, and it should go into your Blue Recycling Bin.

To be sure, it's always best to check the resin identification code on the bottle, as not all plastics are recyclable. Also, please rinse the bottle to remove any residue and remove caps if required by your local recycling rules. This helps ensure it can be reprocessed effectively!


In [39]:
print(len(GEMINI_API_KEY))
print(GEMINI_API_KEY[:6])

53
AQ.Ab8


In [41]:
import time
from google.genai.errors import ClientError

def ask_gemini(prompt):

    while True:

        try:

            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )

            return response.text

        except ClientError as e:

            if "429" in str(e):

                print("Rate limit reached. Waiting 10 seconds...")

                time.sleep(10)

            else:

                raise e